# Лабораторная работа 2
Выполнила Королева Алина, 6401-010302D

## Установка Spark

In [1]:
!pip -q install pyspark

from pyspark.sql import SparkSession, functions as F, Window
import re
from google.colab import files

spark = SparkSession.builder \
    .appName("Lab2") \
    .master("local[*]") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 4.0.2


## Чтение файла постов

In [2]:
posts = spark.read \
    .format('xml') \
    .option('rowTag', 'row') \
    .load('posts_sample.xml')

print(f"Постов: {posts.count()}")

Постов: 46006


## Чтение файла языков

In [3]:
languages = spark.read \
    .option('header', 'true') \
    .csv('programming-languages.csv')

print(f"Языков: {languages.count()}")

Языков: 700


## Фильтрация постов по годам

In [4]:
posts_filtered = posts.select(
    F.year(F.to_timestamp("_CreationDate")).alias("year"),
    F.lower(F.col("_Tags")).alias("tags")
).filter(
    (F.col("year") >= 2010) & (F.col("year") <= 2020) & F.col("tags").isNotNull()
)

## Подготовка списка языков

In [5]:
languages_clean = languages.select(
    F.lower(F.trim("name")).alias("tag")
).dropDuplicates()

print(f"Языков для поиска: {languages_clean.count()}")
languages_clean.show(5)

Языков для поиска: 698
+--------+
|     tag|
+--------+
|  ceylon|
|    hope|
|      m#|
|metafont|
| mortran|
+--------+
only showing top 5 rows


## Парсинг тегов поста

In [6]:
posts_tags = posts_filtered.withColumn(
    "tag",
    F.explode(F.split(F.regexp_replace("tags", r"[<>]", " "), r"\s+"))
).filter(F.col("tag") != "")

## Оставляем только теги языков программирования и год

In [7]:
mentions = posts_tags.join(languages_clean, "tag", "inner") \
    .select("year", "tag")

## Подсчет упоминаний и формирование топ-10

In [8]:
stats = mentions.groupBy("year", "tag").count()

window = Window.partitionBy("year").orderBy(F.desc("count"), F.asc("tag"))

top10 = stats.withColumn("rank", F.row_number().over(window)) \
    .filter(F.col("rank") <= 10) \
    .orderBy("year", "rank")

top10.show(110)

+----+-----------+-----+----+
|year|        tag|count|rank|
+----+-----------+-----+----+
|2010|       java|   52|   1|
|2010|        php|   46|   2|
|2010| javascript|   44|   3|
|2010|     python|   26|   4|
|2010|objective-c|   23|   5|
|2010|          c|   20|   6|
|2010|       ruby|   12|   7|
|2010|     delphi|    8|   8|
|2010|applescript|    3|   9|
|2010|       bash|    3|  10|
|2011|        php|  102|   1|
|2011|       java|   93|   2|
|2011| javascript|   83|   3|
|2011|     python|   37|   4|
|2011|objective-c|   34|   5|
|2011|          c|   24|   6|
|2011|       ruby|   20|   7|
|2011|       perl|    9|   8|
|2011|     delphi|    8|   9|
|2011|       bash|    7|  10|
|2012|        php|  154|   1|
|2012| javascript|  132|   2|
|2012|       java|  124|   3|
|2012|     python|   69|   4|
|2012|objective-c|   45|   5|
|2012|          c|   27|   6|
|2012|       ruby|   27|   7|
|2012|       bash|   10|   8|
|2012|          r|    9|   9|
|2012|        lua|    6|  10|
|2013| jav

##  Сохраняем отчет в Parquet

In [9]:
top10.write.mode("overwrite").parquet("report.parquet")

## Архивируем и скачиваем папку reports.parquet

In [10]:
import shutil
from google.colab import files

# Архивируем папку
shutil.make_archive("report_parquet", "zip", "report.parquet")
files.download("report_parquet.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Остановка spark

In [11]:
spark.stop()